In [1]:
!pip install nltk scikit-learn

In [2]:
import re
import nltk

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [3]:
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [4]:
class PreprocessingModule:

    def __init__(self):
        self.stop_words = set(stopwords.words('english'))

    def transform(self, text):

        # Edge case handling
        if not text.strip():
            return "ERROR: Empty input"

        if len(text.strip()) == 1:
            return "ERROR: Single character input"

        if re.fullmatch(r'[\W\d_]+', text):
            return "ERROR: Only symbols/numbers"

        # Lowercase
        text = text.lower()

        # Remove punctuation
        text = re.sub(r'[^\w\s]', '', text)

        # Tokenization
        words = word_tokenize(text)

        # Stopword removal
        filtered_words = [
            word for word in words
            if word not in self.stop_words
        ]

        return " ".join(filtered_words)

In [5]:
class VectorizerModule:

    def __init__(self):
        self.vectorizer = TfidfVectorizer()

    def fit(self, corpus):
        self.corpus_vectors = self.vectorizer.fit_transform(corpus)

    def transform(self, query):
        return self.vectorizer.transform([query])

In [6]:
class Pipeline:

    def __init__(self):
        self.preprocessor = PreprocessingModule()
        self.vectorizer = VectorizerModule()

    def run(self, query, corpus):

        processed_corpus = []

        for doc in corpus:
            processed_doc = self.preprocessor.transform(doc)
            processed_corpus.append(processed_doc)

        processed_query = self.preprocessor.transform(query)

        # Handle edge case errors
        if processed_query.startswith("ERROR"):
            return processed_query

        # Fit vectorizer
        self.vectorizer.fit(processed_corpus)

        # Transform query
        query_vector = self.vectorizer.transform(processed_query)

        # Similarity calculation
        similarities = cosine_similarity(
            query_vector,
            self.vectorizer.corpus_vectors
        )[0]

        ranked_results = sorted(
            zip(corpus, similarities),
            key=lambda x: x[1],
            reverse=True
        )

        return ranked_results

In [7]:
corpus = [
    "Artificial intelligence is transforming healthcare",
    "Machine learning improves recommendation systems",
    "Deep learning powers image recognition",
    "Python is widely used in data science",
    "Natural language processing helps chatbots",
    "Football is a popular sport worldwide",
    "Basketball players require strong teamwork",
    "Healthy food improves body fitness",
    "Exercise is important for mental health",
    "Climate change affects global weather",
    "Space exploration advances scientific discovery",
    "Cybersecurity protects digital systems",
    "Cloud computing enables scalable applications",
    "Students use AI tools for learning",
    "Data preprocessing improves machine learning accuracy"
]

In [8]:
pipeline = Pipeline()

queries = [
    "AI in education",
    "machine learning accuracy",
    "sports teamwork",
    "weather and climate",
    "healthy lifestyle"
]

for query in queries:

    print("=" * 60)
    print("QUERY:", query)
    print("=" * 60)

    results = pipeline.run(query, corpus)

    for doc, score in results[:5]:
        print(f"Score: {score:.4f} | {doc}")

    print()

QUERY: AI in education
Score: 0.4717 | Students use AI tools for learning
Score: 0.0000 | Artificial intelligence is transforming healthcare
Score: 0.0000 | Machine learning improves recommendation systems
Score: 0.0000 | Deep learning powers image recognition
Score: 0.0000 | Python is widely used in data science

QUERY: machine learning accuracy
Score: 0.6988 | Data preprocessing improves machine learning accuracy
Score: 0.4384 | Machine learning improves recommendation systems
Score: 0.1553 | Deep learning powers image recognition
Score: 0.1553 | Students use AI tools for learning
Score: 0.0000 | Artificial intelligence is transforming healthcare

QUERY: sports teamwork
Score: 0.4472 | Basketball players require strong teamwork
Score: 0.0000 | Artificial intelligence is transforming healthcare
Score: 0.0000 | Machine learning improves recommendation systems
Score: 0.0000 | Deep learning powers image recognition
Score: 0.0000 | Python is widely used in data science

QUERY: weather and

In [9]:
edge_cases = [
    "",
    "a",
    "@@@@12345"
]

for case in edge_cases:

    print("=" * 50)
    print("INPUT:", repr(case))

    result = pipeline.run(case, corpus)

    print("OUTPUT:", result)
    print()

INPUT: ''
OUTPUT: ERROR: Empty input

INPUT: 'a'
OUTPUT: ERROR: Single character input

INPUT: '@@@@12345'
OUTPUT: ERROR: Only symbols/numbers

